# 의료영상에서 AI는 무엇을 볼까? 🧠
### 진짜 MRI 사진에서 종양을 **찾고**, 사진을 **그리는** AI를 직접 만들어 봅니다

오늘 20분 동안 할 일:

1. **진짜 뇌 MRI 데이터 열어 보기** → 환자 233명, 3,064장의 공개 데이터. 컴퓨터에게 사진은 숫자표
2. **사람이 규칙 정하기 vs AI가 예시에서 배우기** → "밝으면 종양"부터 규칙을 4개까지 겹쳐 봅니다
3. **종양 찾기 (Segmentation)** → AI가 점점 배우는 과정을 단계별로 봅니다
4. **MRI 그리기 (Generation)** → 아무것도 없는 노이즈에서 AI가 새로운 MRI를 만들어 냅니다 (diffusion)
5. **챗봇 AI(ChatGPT·Gemini)에게** 우리 결과를 보여주고 설명을 들어봅니다

> 💡 모든 코드는 이 노트북 안에서 **한 칸씩 직접 실행**합니다. `Shift + Enter`를 누르면 한 칸(셀)이 실행됩니다.

> ☁️ **Google Colab용 버전입니다.** 아래 셀을 먼저 실행하면 데이터와 미리 학습한 모델을 자동으로 받아옵니다 (약 150MB, 1~2분). 라이브러리와 한글 폰트는 그다음 준비 셀이 알아서 챙깁니다.
> 메뉴 `런타임 → 모두 실행`으로 한 번에 돌릴 수도 있습니다.

In [ ]:
# Colab 준비: GitHub에서 데이터·모델 받아오기 + 한글 폰트 설치
REPO_URL = "https://github.com/pjh1023/mri-ai-demo"      # 데이터·모델이 올라가 있는 GitHub 저장소

import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("mri_demo"):
        !git clone --depth 1 {REPO_URL} mri_demo
    %cd mri_demo
print("준비 끝. 현재 폴더:", os.getcwd())

In [ ]:
# 준비 1: 필요한 라이브러리가 없으면 설치합니다 (이미 있으면 그냥 지나감. 처음 한 번만 몇 분 걸릴 수 있음)
import importlib.util, subprocess, sys, platform
for pkg in ["numpy", "scipy", "matplotlib", "torch"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"'{pkg}' 설치 중...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

# 준비 2: 도구(라이브러리)를 불러옵니다
import numpy as np                 # 숫자 계산
import matplotlib.pyplot as plt    # 그림 그리기
import torch, torch.nn as nn       # 인공신경망(AI) 만들기
import torch.nn.functional as F
import time, os

torch.manual_seed(0); np.random.seed(0)   # 매번 같은 결과가 나오도록 고정

# 준비 3: 그림에 한글이 나오도록 폰트 설정 (Windows: 맑은 고딕 / Mac: AppleGothic / Linux·Colab: 나눔고딕을 설치)
import matplotlib.font_manager as fm
KOREAN_FONTS = ["Malgun Gothic", "AppleGothic", "NanumGothic"]
def korean_font():
    return next((f for f in KOREAN_FONTS if any(f == x.name for x in fm.fontManager.ttflist)), None)
if korean_font() is None and platform.system() == "Linux":
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null 2>&1 || sudo apt-get -qq install -y fonts-nanum", shell=True)
    for ttf in fm.findSystemFonts(["/usr/share/fonts/truetype/nanum"]):
        fm.fontManager.addfont(ttf)
if korean_font():
    plt.rcParams["font.family"] = korean_font()
else:
    print("⚠ 한글 폰트를 찾지 못했습니다. 그림의 한글이 네모로 보일 수 있습니다 (코드 실행에는 문제 없음).")
plt.rcParams["axes.unicode_minus"] = False
print("준비 완료!  PyTorch", torch.__version__, "/ 폰트:", korean_font())

---
## 1. 진짜 뇌 MRI 데이터 열어 보기

오늘은 처음부터 **진짜 환자 MRI**를 씁니다. 중국 난팡의과대학 연구팀이 공개한 데이터(Cheng 외, 2015)로,
환자 **233명**의 뇌 MRI **3,064장**에 의사가 직접 종양 테두리를 그려 놓았습니다. (연구·교육용으로 공개된 CC BY 4.0 데이터)

컴퓨터가 빨리 처리할 수 있게 **3,064장 전부**를 128×128 크기로 줄여서 가져왔습니다.

In [ ]:
data = np.load("data/brain_mri_128.npz")
images  = data["image"] / 255.0            # MRI 사진 3064장 (밝기 0~1)
masks   = data["mask"].astype(np.float32)  # 정답: 의사가 그린 종양 자리 (1=종양, 0=아님)
types   = data["label"]                    # 종양 종류: 1 수막종, 2 신경교종, 3 뇌하수체종양
is_test = data["is_test"]                  # True = '처음 보는 환자' (AI 시험용으로 따로 둠)
TYPE_NAME = {1: "수막종", 2: "신경교종", 3: "뇌하수체종양"}

print("사진 묶음 크기:", images.shape, "→ 3064장, 각 128x128 픽셀")
print("종양 종류별 장수:", {TYPE_NAME[t]: int((types == t).sum()) for t in (1, 2, 3)})
print("종양이 차지하는 비율: 전체 픽셀의", f"{masks.mean()*100:.1f}%")

In [ ]:
# 원본에 가까운 해상도(256x256)로 몇 장 구경하기 — 빨간 선이 의사가 그린 종양 테두리
ex = np.load("data/examples_256.npz")
plt.figure(figsize=(13, 4.6))
for i in range(6):
    plt.subplot(1, 6, i+1); plt.imshow(ex["image"][i], cmap="gray"); plt.axis("off")
    plt.contour(ex["mask"][i], levels=[0.5], colors="red", linewidths=1.2)
    plt.title(TYPE_NAME[int(ex["label"][i])], fontsize=10)
plt.show()

### 컴퓨터에게 이 사진은 그냥 **숫자표**입니다
사람은 사진을 보고 "여기가 뇌, 여기가 종양"을 바로 알지만, 컴퓨터가 받는 것은 **128×128 = 16,384개의 숫자**뿐입니다.
밝은 곳은 1에 가까운 숫자, 어두운 곳은 0에 가까운 숫자. 그래서 컴퓨터가 종양을 찾으려면 **숫자로 된 규칙**이 필요합니다.
종양 가장자리를 확대해서 숫자를 직접 적어 보겠습니다.

In [ ]:
i = 500                                                        # 사진 한 장 고르기
img, mask = images[i], masks[i]
row   = np.argmax(mask.sum(1))                                 # 종양이 가장 넓은 줄
right = np.where(mask[row] > 0)[0].max()                       # 그 줄에서 종양의 오른쪽 끝
y0, x0 = row - 4, right - 3
patch = img[y0:y0+8, x0:x0+8]                                  # 종양 경계 8x8 픽셀

plt.figure(figsize=(8, 3.6))
plt.subplot(1, 2, 1); plt.imshow(img, cmap="gray"); plt.axis("off"); plt.title("빨강 = 종양, 노란 네모를 확대하면 →", fontsize=9)
plt.contour(mask, levels=[0.5], colors="red", linewidths=1)
plt.gca().add_patch(plt.Rectangle((x0-0.5, y0-0.5), 8, 8, fill=False, color="yellow", lw=2))
plt.subplot(1, 2, 2); plt.imshow(patch, cmap="gray", vmin=0, vmax=1); plt.axis("off"); plt.title("컴퓨터가 보는 것: 숫자 64개", fontsize=9)
for (r, c), v in np.ndenumerate(patch):
    plt.text(c, r, f"{v:.1f}", ha="center", va="center", color="red" if v > 0.6 else "yellow", fontsize=8)
plt.show()

---
## 2. 사람이 규칙을 정하기 vs AI가 예시에서 배우기

### 방법 1 — 사람이 규칙을 정하기: "밝은 곳이 종양이야!"
조영제를 맞으면 종양이 밝게 보입니다. 그러면 숫자가 0.6보다 큰 곳을 찍으면 될까요?

먼저 결과를 그려 주는 도우미 함수와, **얼마나 잘 맞았는지 점수(일치도)** 를 재는 함수를 만들어 둡니다.
일치도(Dice)는 "내 답과 정답이 겹치는 정도"로, 의료영상 AI 평가에 실제로 쓰는 점수입니다. (0% = 하나도 안 겹침, 100% = 완벽)

In [ ]:
def draw_one(img, mask=None, pred=None):
    "사진 한 장 + 빨강 선(정답) + 하늘색(AI/규칙이 '종양'이라고 한 곳 = 확률 50% 초과)"
    plt.imshow(img, cmap="gray", vmin=0, vmax=1); plt.axis("off")
    if pred is not None:
        p = np.asarray(pred, dtype=float)
        plt.contourf(p, levels=[0.5, 1.01], colors=["cyan"], alpha=0.35)       # 50%를 넘긴 곳을 연하게 채우고
        plt.contour(p, levels=[0.5], colors="cyan", linewidths=1.3)            # 테두리를 선으로
    if mask is not None:
        plt.contour(mask, levels=[0.5], colors="red", linewidths=1.5)

def show_row(images, masks=None, preds=None, titles=None, size=2.3):
    "이미지 여러 장을 한 줄로"
    n = len(images); plt.figure(figsize=(size*n, size+0.3))
    for i in range(n):
        plt.subplot(1, n, i+1)
        draw_one(images[i], None if masks is None else masks[i], None if preds is None else preds[i])
        if titles is not None: plt.title(titles[i], fontsize=9)
    plt.show()

def show_with_maps(images, masks, preds, label, size=2.3):
    "윗줄: 사진 + 정답 + AI 답,  아랫줄: AI가 픽셀마다 매긴 '종양일 확률' 지도 (밝을수록 확신)"
    n = len(images); plt.figure(figsize=(size*n, 2*size+0.5))
    for i in range(n):
        plt.subplot(2, n, i+1); draw_one(images[i], masks[i], preds[i])
        plt.title(f"{label}: 일치도 {dice(preds[i], masks[i]):.0f}%", fontsize=9)
        plt.subplot(2, n, n+i+1); plt.imshow(preds[i], cmap="magma", vmin=0, vmax=1); plt.axis("off")
        plt.title(f"확률 지도: {preds[i].min()*100:.0f}% ~ {preds[i].max()*100:.0f}%", fontsize=9)
    plt.show()

def dice(pred, truth):
    "일치도: 답과 정답이 겹치는 정도 (0% ~ 100%)"
    p, t = np.asarray(pred) > 0.5, np.asarray(truth) > 0.5
    return 100 * 2 * (p & t).sum() / (p.sum() + t.sum() + 1e-6)

sample, answer = images[500], masks[500]
guess = sample > 0.6                                  # 규칙 ①: 밝기가 0.6보다 크면 종양!
show_row([sample, sample], masks=[answer, answer], preds=[None, guess],
         titles=["빨강 = 의사가 그린 정답", f"하늘색 = '밝으면 종양' 규칙 → 일치도 {dice(guess, answer):.0f}%"], size=3)

😅 두개골, 피부, 혈관… 밝은 건 너무 많습니다. 그러면 규칙을 더 보태 볼까요?

- ② "두개골은 머리의 바깥 테두리니까, 테두리에서 10픽셀 안쪽만 보자"
- ③ "종양은 머리 가운데쯤 있으니까, 중심에서 35픽셀 안만 보자"
- ④ "종양은 덩어리 하나니까, 가장 큰 덩어리 하나만 남기자"

규칙을 하나씩 겹쳐 가면서 **같은 사진**에 적용해 보고, **시험용 사진 573장 전체**의 평균 일치도도 재 봅니다.

In [ ]:
from scipy.ndimage import binary_fill_holes, binary_erosion, label as find_blobs

def head_outline(img):
    "머리 윤곽 찾기: 밝기 0.1보다 밝은 덩어리 중 가장 큰 것 (구멍은 메움)"
    blobs, n = find_blobs(img > 0.1)
    biggest = np.argmax(np.bincount(blobs.ravel())[1:]) + 1
    return binary_fill_holes(blobs == biggest)

def rules(img):
    "규칙 ①~④를 차례로 겹쳐 적용 → 결과 4개"
    head = head_outline(img)
    r1 = img > 0.6                                                      # ① 밝으면 종양
    r2 = r1 & binary_erosion(head, iterations=10)                       # ② + 머리 테두리(두개골·피부)에서 10픽셀 안쪽만
    cy, cx = np.argwhere(head).mean(0); yy, xx = np.mgrid[0:128, 0:128]
    r3 = r2 & ((yy - cy)**2 + (xx - cx)**2 < 35**2)                     # ③ + 머리 중심에서 35픽셀 안만
    blobs, n = find_blobs(r3)
    r4 = (blobs == np.argmax(np.bincount(blobs.ravel())[1:]) + 1) if n else r3   # ④ + 가장 큰 덩어리 하나만
    return [r1, r2, r3, r4]

RULE_NAMES = ["① 밝으면", "② +테두리 빼기", "③ +가운데만", "④ +큰 덩어리만"]
results = rules(sample)
show_row([sample]*5, masks=[answer]*5, preds=[None] + results,
         titles=["정답"] + [f"{n}: {dice(r, answer):.0f}%" for n, r in zip(RULE_NAMES, results)], size=2.5)

test_imgs, test_masks = images[is_test], masks[is_test]               # 시험용 573장에 네 규칙을 전부 적용
scores = np.array([[dice(r, m) for r in rules(x)] for x, m in zip(test_imgs, test_masks)])
for name, s in zip(RULE_NAMES, scores.mean(0)):
    print(f"{name:<10s} 시험용 573장 평균 일치도 {s:4.0f}%")

🤔 규칙을 4개나 겹쳤는데도 평균 점수는 한참 낮습니다. 사진 한 장에 맞춰 규칙을 다듬어도 **다른 사진에서는 어긋납니다.**

### 왜? "가운데쯤", "이 정도 크기", "테두리에서 10픽셀"이 사람마다 다르기 때문입니다
MRI는 머리를 위·앞·옆 세 방향으로 잘라 찍습니다. **같은 방향으로 찍은 사진끼리** 모아서 머리 윤곽(노란 선)을 한 곳에 겹쳐 보면,
같은 방향인데도 사람마다 머리 크기·모양·기울기가 전부 다릅니다. 종양의 위치·크기·밝기도 제각각입니다.

In [ ]:
VIEWS = {"위에서 자른 단면": [8, 96, 184, 1320, 1856, 2112, 2176, 2360],          # 방향별로 서로 다른 사진 8장씩
         "앞에서 자른 단면": [1088, 1120, 1472, 1504, 1552, 1600, 2448, 2496],
         "옆에서 자른 단면": [528, 592, 640, 696, 1184, 1696, 1760, 2848]}
fig, axes = plt.subplots(3, 9, figsize=(15, 5.4))
for r, (view, pick) in enumerate(VIEWS.items()):
    for k, i in enumerate(pick):
        head = head_outline(images[i]); ax = axes[r, k]
        ax.imshow(images[i], cmap="gray"); ax.axis("off")
        ax.contour(head, levels=[0.5], colors="yellow", linewidths=1.0)
        ax.contour(masks[i], levels=[0.5], colors="red", linewidths=0.8)
        ax.set_title(f"넓이 {int(head.sum()):,}", fontsize=7, pad=2)
        if k == 0: ax.text(-6, 64, view, fontsize=9, rotation=90, va="center", ha="right")
    ax = axes[r, 8]                                                     # 맨 오른쪽: 8장의 윤곽을 한 곳에 겹치기
    for k, i in enumerate(pick):
        ax.contour(head_outline(images[i]), levels=[0.5], colors=[plt.cm.tab10(k)], linewidths=1.2)
    ax.set_xlim(0, 128); ax.set_ylim(128, 0); ax.set_aspect("equal"); ax.axis("off"); ax.set_title("8장 겹치면", fontsize=8, pad=2)
plt.tight_layout(); plt.show()
for view, pick in VIEWS.items():
    print(f"{view}  종양 크기(픽셀 수):", [int(masks[i].sum()) for i in pick])

### 방법 2 — AI에게 **정답 예시를 많이 보여주고 스스로 규칙을 찾게** 하기

규칙을 사람이 적는 대신, "문제(사진)와 정답(종양 자리)" 묶음을 수천 개 보여주고 AI가 **스스로 규칙을 찾아내게** 합니다.
사진을 두 묶음으로 나눕니다.
- **학습용** (환자의 80%): 가르칠 때 쓰는 사진
- **시험용** (나머지 환자 20%): AI가 **한 번도 본 적 없는 환자** 사진으로 실력 확인하기 (방금 규칙에도 썼던 573장)

In [ ]:
X  = torch.tensor(images[~is_test], dtype=torch.float32)[:, None]   # 학습용 사진   (장수, 1, 128, 128)
Y  = torch.tensor(masks[~is_test])[:, None]                         # 학습용 정답
Xt = torch.tensor(images[is_test], dtype=torch.float32)[:, None]    # 시험용 사진
Yt = torch.tensor(masks[is_test])[:, None]                          # 시험용 정답
N = len(X)
print("학습용:", N, "장   시험용:", len(Xt), "장   사진 한 장 크기:", tuple(X.shape[1:]))

---
## 3. 종양 찾기 AI 만들기 (Segmentation)

**목표:** 사진을 넣으면, 픽셀마다 "종양일 확률"을 찍어주는 프로그램

의료영상 분야에서 가장 유명한 모델 **U-Net**입니다.
사진을 점점 **줄여서 큰 그림을 보고(Encoder)** → 다시 **키우면서 정확한 위치를 찍습니다(Decoder)**.

겁먹지 마세요. 핵심은 하나입니다: **`Conv2d`(작은 돋보기로 사진을 훑는 층)를 여러 겹 쌓은 것.**

In [ ]:
def block(cin, cout):
    "작은 돋보기(3x3)로 사진을 훑는 층 2개"
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU())

class UNet(nn.Module):
    def __init__(self, w=32):
        super().__init__()
        self.enc = nn.ModuleList([block(1, w), block(w, 2*w), block(2*w, 4*w), block(4*w, 8*w)])   # 줄이며 보기: 128→64→32→16
        self.dec = nn.ModuleList([block(8*w+4*w, 4*w), block(4*w+2*w, 2*w), block(2*w+w, w)])      # 키우며 찍기: 16→32→64→128
        self.out = nn.Conv2d(w, 1, 1)
        nn.init.constant_(self.out.bias, -4)      # 시작값: "종양은 드물다"(확률 약 2%)는 것만 알고 출발
    def forward(self, x):
        skips = []
        for i, enc in enumerate(self.enc):
            x = enc(x if i == 0 else F.max_pool2d(x, 2)); skips.append(x)       # 절반으로 줄이며 보기
        for dec, skip in zip(self.dec, skips[-2::-1]):
            x = dec(torch.cat([F.interpolate(x, scale_factor=2), skip], 1))     # 2배로 키우며, 줄이기 전 모습도 같이 보기
        return self.out(x)

model = UNet()
print("이 AI가 스스로 조절할 숫자(파라미터) 개수:", f"{sum(p.numel() for p in model.parameters()):,}")

### 학습 전 — 아직 아무것도 모르는 AI
시험용 사진(처음 보는 환자) 5장을 보여줍니다. AI는 픽셀마다 "종양일 확률"을 매기고, **50%를 넘긴 곳**이 AI의 답(하늘색)입니다.
아랫줄은 그 확률을 그대로 그린 **확률 지도**입니다 (밝을수록 "종양"이라고 확신).

In [ ]:
def predict(model, imgs):
    "모델에게 사진을 보여주고 '종양 확률 지도'(0~1)를 받는다"
    model.eval()
    with torch.no_grad():
        return torch.cat([torch.sigmoid(model(imgs[i:i+64])) for i in range(0, len(imgs), 64)])[:, 0].numpy()

show = [547, 92, 310, 322, 249]                   # 시험용 사진 번호 5개 (종양 크기·찍은 방향이 서로 다른 사진)
show_imgs, show_masks = Xt[show, 0].numpy(), Yt[show, 0].numpy()
preds = predict(model, Xt[show])
show_with_maps(show_imgs, show_masks, preds, label="학습 전")
print(f"확률 지도가 온통 한 가지 색: 모든 픽셀에 '{preds.min()*100:.0f}~{preds.max()*100:.0f}%'라고 답함 → 어디가 종양인지 전혀 모르는 상태 (50%를 넘긴 곳이 없으니 하늘색도 없음)")

### 학습 — "틀리면 조금 고치기"를 수백 번 반복

1. 사진을 보여주고 답을 받는다
2. 정답과 비교해서 **얼마나 틀렸는지(loss)** 숫자로 잰다
3. 덜 틀리는 방향으로 파라미터를 **아주 조금** 고친다 (`optimizer.step()`)
4. 반복!

In [ ]:
def loss_fn(logits, truth):
    "얼마나 틀렸나를 재는 자: 픽셀마다 틀린 정도 + 전체 모양이 안 겹치는 정도"
    p = torch.sigmoid(logits)                                    # 0~1 사이 '종양일 확률'
    pixel_wrong = F.binary_cross_entropy_with_logits(logits, truth)
    not_overlap = 1 - 2 * (p * truth).sum() / (p.sum() + truth.sum() + 1)
    return pixel_wrong + not_overlap

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)      # 파라미터를 고쳐 주는 도구
EPOCHS, BATCH, N_LIVE = 1, 16, 300                              # 현장에서는 300장만 1바퀴, 한 번에 16장씩
model.train(); history = []; t0 = time.time()
for epoch in range(EPOCHS):
    perm = torch.randperm(N)[:N_LIVE]
    for i in range(0, N_LIVE, BATCH):
        idx = perm[i:i+BATCH]
        loss = loss_fn(model(X[idx]), Y[idx])     # 1~2. 답을 내고 정답과 비교
        optimizer.zero_grad(); loss.backward()    # 3. 어디를 고칠지 계산
        optimizer.step()                          #    조금 고치기
        history.append(loss.item())
    print(f"{epoch+1}바퀴 끝   틀린 정도(loss) = {loss.item():.3f}   ({time.time()-t0:.0f}초 경과)")

plt.figure(figsize=(5, 2.5)); plt.plot(history); plt.xlabel("고친 횟수"); plt.ylabel("틀린 정도"); plt.title("점점 덜 틀린다!"); plt.show()

### 짧게 학습한 후 — 같은 사진을 다시 보여주면?

In [ ]:
preds = predict(model, Xt[show])
show_with_maps(show_imgs, show_masks, preds, label="짧게 학습")

🤔 확률 지도에 사진의 밝은 부분이 떠오르기 시작했습니다. 30번 고친 것만으로 "밝은 곳이 수상하다"까지는 배웠지만, 아직 두개골과 종양을 헷갈립니다. **진짜 데이터는 어렵습니다.**

### 학습을 계속하면 AI는 어떻게 변할까? — 학습 과정을 단계별로 보기
실제 연구에서는 GPU로 몇 시간~며칠 학습합니다. 오늘은 시간이 없으니,
**똑같은 코드로 집에서 미리 전체 2,500장을 300바퀴 돌리면서, 중간중간 저장해 둔 파라미터**를 차례로 불러와 보겠습니다.
(1바퀴 = 학습용 사진 2,491장을 한 번씩 다 보는 것)

In [ ]:
stage_files = {1: "models/stages/ep1.pt", 5: "models/stages/ep5.pt", 20: "models/stages/ep20.pt", 300: "models/segmenter.pt"}

stage_mean = {}
plt.figure(figsize=(2.3*5, 2.2*len(stage_files)))
for r, (ep, path) in enumerate(stage_files.items()):
    model.load_state_dict(torch.load(path, weights_only=True))          # 그 시점의 파라미터 불러오기
    preds = predict(model, Xt[show])
    stage_mean[ep] = np.mean([dice(p, m) for p, m in zip(predict(model, Xt[::4]), Yt[::4, 0].numpy())])   # 시험용 사진 4장 중 1장(144장) 평균
    for c in range(5):
        plt.subplot(len(stage_files), 5, r*5 + c + 1); draw_one(show_imgs[c], show_masks[c], preds[c])
        plt.title(f"{ep}바퀴: 일치도 {dice(preds[c], show_masks[c]):.0f}%", fontsize=9)
plt.tight_layout(); plt.show()

plt.figure(figsize=(5, 2.6))
plt.plot(range(len(stage_mean)), list(stage_mean.values()), "o-")
plt.xticks(range(len(stage_mean)), [f"{ep}바퀴" for ep in stage_mean]); plt.ylabel("평균 일치도 (%)"); plt.ylim(0, 100)
plt.title("시험용 사진 144장 평균: 오래 배울수록 올라간다"); plt.grid(alpha=0.3); plt.show()

처음엔 밝은 곳이면 다 찍다가(두개골까지), 점점 **두개골·혈관은 피하고 종양만** 찍게 됩니다.
AI가 2,500장의 예시를 보고 "종양은 이런 모양·위치·밝기"라는 규칙을 **스스로** 찾아낸 것입니다. 우리가 적어 준 규칙은 하나도 없습니다.

### 최종 모델의 실력

In [ ]:
model.load_state_dict(torch.load("models/segmenter.pt", weights_only=True))   # 300바퀴 학습한 최종 파라미터
all_preds = predict(model, Xt)                                                 # 시험용 573장 전부 예측
all_dice = np.array([dice(p, m) for p, m in zip(all_preds, Yt[:, 0].numpy())])
found = np.mean([((p > 0.5) & (m > 0.5)).any() for p, m in zip(all_preds, Yt[:, 0].numpy())])

show_row(show_imgs, masks=show_masks, preds=all_preds[show],
         titles=[f"300바퀴: 일치도 {all_dice[i]:.0f}%" for i in show])
print(f"처음 보는 환자 사진 {len(Xt)}장:  평균 일치도 {all_dice.mean():.0f}%   중간값 {np.median(all_dice):.0f}%   종양 위치를 찾아낸 사진 {found*100:.0f}%")
print("(참고) 사람이 만든 규칙 ④의 평균 일치도:", f"{scores.mean(0)[3]:.0f}%")

### 🙋 직접 해보기: 시험용 사진 번호를 골라 보세요 (0 ~ 572)

In [ ]:
i = 42        # ← 0~572 사이 숫자를 바꿔 보세요!
p = all_preds[i]
show_row([Xt[i, 0], Xt[i, 0], Xt[i, 0]], masks=[None, Yt[i, 0], Yt[i, 0]], preds=[None, None, p],
         titles=["AI가 받은 사진", "빨강 = 의사의 정답", f"하늘색 = AI 답 (일치도 {all_dice[i]:.0f}%)"], size=3)

### 😬 AI도 틀립니다 — 가장 못 맞춘 사진 3장

In [ ]:
worst = np.argsort(all_dice)[:3]
show_row(Xt[worst, 0], masks=Yt[worst, 0], preds=all_preds[worst],
         titles=[f"{i}번: 일치도 {all_dice[i]:.0f}%" for i in worst], size=3)

> 💬 작거나 희미한 종양, 특이한 위치의 종양은 놓치거나 다른 곳을 찍습니다. 그래서 **의료 AI는 의사를 돕는 도구**이지 의사를 대신하지 않습니다.
> AI가 찍은 것을 의사가 확인하고, 틀린 경우를 모아 다시 가르칩니다.

---
## 4. MRI 그리기 (Generation) — 노이즈에서 새 MRI를 만들어 내기

지금까지는 **사진 → 종양 지도**였습니다 (탐정 AI). 이번엔 AI가 **아무것도 없는 데서 사진을 그리게** 합니다 (화가 AI).

요즘 그림 그리는 AI(Stable Diffusion, 미드저니, 나노바나나 등)가 쓰는 **diffusion(확산)** 방식입니다. 아이디어는 단순합니다.

1. 진짜 사진에 노이즈(지지직)를 조금씩 섞어서 완전히 망가뜨린다 → 이건 누구나 할 수 있다
2. AI에게 **거꾸로** 가르친다: "노이즈 낀 사진을 줄 테니, 섞여 있는 노이즈가 뭔지 맞혀 봐"
3. 다 배우고 나면: **완전한 노이즈**에서 출발해 조금씩 노이즈를 걷어내면 → 세상에 없던 새 MRI가 나온다

In [ ]:
from diffusion_model import DiffusionUNet, NoiseSchedule   # 모델 정의는 파일에 따로 두었습니다 (U-Net + "몇 단계인지" 입력)
DEV = "cuda" if torch.cuda.is_available() else "cpu"        # 이 파트는 계산이 많아서 GPU가 있으면 GPU를 씁니다 (없어도 됨)

X128 = (torch.tensor(images, dtype=torch.float32)[:, None] * 2 - 1).to(DEV)   # 그리기용: 밝기를 -1~1로
schedule = NoiseSchedule(T=1000); schedule.alpha_bar = schedule.alpha_bar.to(DEV)   # 1000단계에 걸쳐 노이즈를 섞는 표

# 1단계: 진짜 사진에 노이즈를 점점 더 섞어 보기
x0 = X128[500:501]
steps = [0, 50, 200, 400, 700, 999]
plt.figure(figsize=(12, 2.3))
for k, t in enumerate(steps):
    noisy = schedule.add_noise(x0, torch.tensor([t], device=DEV), torch.randn_like(x0))
    plt.subplot(1, 6, k+1); plt.imshow(noisy[0, 0].cpu(), cmap="gray", vmin=-1, vmax=1); plt.axis("off"); plt.title(f"t = {t}", fontsize=9)
plt.suptitle("오른쪽으로 갈수록 노이즈가 많이 섞인다 (t=999면 완전한 노이즈)", fontsize=9); plt.show()

### 2단계: AI에게 "섞인 노이즈 맞히기"를 가르치기
학습 루프는 종양 찾기 때와 똑같은 모양입니다. 다만 **정답이 노이즈**입니다.

In [ ]:
painter = DiffusionUNet().to(DEV)                             # 입력: 노이즈 낀 사진 + 단계 t → 출력: 섞인 노이즈 추정
print("화가 AI의 파라미터 개수:", f"{sum(p.numel() for p in painter.parameters()):,}")
opt = torch.optim.Adam(painter.parameters(), lr=2e-4)
t0 = time.time(); history = []
for step in range(10):                                        # 현장에서는 10번만 (CPU 약 20초)
    x0 = X128[torch.randint(0, len(X128), (4,), device=DEV)]  # 진짜 사진 4장
    t = torch.randint(0, 1000, (4,), device=DEV)              # 사진마다 아무 단계나 고르기
    noise = torch.randn_like(x0)                              # 섞을 노이즈 (이게 정답!)
    noisy = schedule.add_noise(x0, t, noise)                  # 노이즈 섞기
    loss = ((painter(noisy, t) - noise) ** 2).mean()          # AI가 맞힌 노이즈 vs 진짜 노이즈
    opt.zero_grad(); loss.backward(); opt.step()
    history.append(loss.item())
print(f"10번 고침   틀린 정도 {history[0]:.3f} → {history[-1]:.3f}   ({time.time()-t0:.0f}초)")

painter.load_state_dict(torch.load("models/diffusion.pt", weights_only=True, map_location=DEV))   # 집에서 150,000번 미리 고쳐 둔 것 불러오기
painter.eval(); print("미리 학습한 화가 AI 불러오기 완료")

### 3단계: 완전한 노이즈에서 출발해서 새 MRI 만들기 🎨
이 사진들은 **데이터에 없는, 세상에 존재하지 않는 뇌**입니다.

In [ ]:
torch.manual_seed(7)                                                   # ← 숫자를 바꾸면 다른 뇌가 나옵니다
t0 = time.time()
fakes, frames = schedule.sample(painter, n=6, size=128, steps=40, keep_every=6)   # 40번에 걸쳐 노이즈 걷어내기 (CPU 약 20초)
fakes, frames = fakes.cpu(), [f.cpu() for f in frames]
plt.figure(figsize=(13, 2.4))
for i in range(6):
    plt.subplot(1, 6, i+1); plt.imshow(fakes[i, 0], cmap="gray", vmin=-1, vmax=1); plt.axis("off")
plt.suptitle(f"AI가 만든 MRI 6장 (세상에 없는 뇌)   —   {time.time()-t0:.0f}초 걸림", fontsize=10); plt.show()

In [ ]:
# 노이즈가 걷히는 과정 보기 (첫 번째 사진)
plt.figure(figsize=(13, 2.2))
for k, fr in enumerate(frames):
    plt.subplot(1, len(frames), k+1); plt.imshow(fr[0, 0], cmap="gray", vmin=-1, vmax=1); plt.axis("off")
    plt.title("시작(노이즈)" if k == 0 else ("완성" if k == len(frames)-1 else f"{k*6}/40"), fontsize=8)
plt.show()

### 🔁 두 AI를 연결하면? — 화가 AI가 만든 뇌를 탐정 AI에게 보여주기
만들어진 뇌에도 종양이 있을까요? 하늘색이 탐정 AI가 "종양"이라고 찍은 곳입니다.

In [ ]:
fake_imgs = ((fakes + 1) / 2)                                   # 밝기를 탐정 AI가 배운 0~1로 맞추기
found_fake = predict(model, fake_imgs)
show_row(fake_imgs[:, 0].numpy(), preds=found_fake, titles=[f"종양 후보 {int((f > 0.5).sum())}픽셀" for f in found_fake], size=2.1)

> 💬 만든 뇌 대부분에서 탐정 AI가 밝은 덩어리를 찾아냅니다. 학습 데이터가 **전부 종양 환자 사진**이라, 화가 AI는 "뇌 MRI에는 밝은 덩어리가 하나 있다"까지 배워서 종양까지 그려 넣은 것입니다.
> 데이터가 AI를 만듭니다. 건강한 사람 사진으로 가르쳤다면 종양 없는 뇌를 그렸을 겁니다.
> (단, 이 뇌에는 의사가 그린 정답이 없으니 하늘색이 정말 종양인지는 아무도 모릅니다.)
>
> 💬 **왜 사진을 만들까?** 희귀한 병은 사진이 너무 적어서 AI가 만든 사진으로 학습 데이터를 늘리고,
> 환자 개인정보 없이 연구·교육용 사진을 만들고, "치료 후 모습"을 미리 그려 보기도 합니다.
>
> 💬 오늘 모델은 파라미터 4,550만 개, 128×128 크기입니다. Stable Diffusion은 파라미터 **10억 개**, 같은 원리로 1024×1024 그림을 그립니다.

---
## 5. 챗봇 AI(ChatGPT·Gemini)에게 우리 결과를 보여주고 물어보기

요즘 가장 유명한 AI는 ChatGPT, Gemini, Claude 같은 **대형 언어 모델(LLM)** 입니다.
LLM은 글뿐 아니라 **사진도 볼 수 있습니다.** 우리가 만든 결과 그림을 보여주고 평가를 들어봅시다.

아래 셀은 결과 그림 2장을 `output/` 폴더에 파일로 저장하고, 물어볼 질문을 출력합니다.
그다음은 **브라우저**에서 합니다: 챗봇 사이트를 열고 → 그림을 붙여 넣고 → 질문을 붙여 넣기.

In [ ]:
os.makedirs("output", exist_ok=True)

fig = plt.figure(figsize=(9, 3.2))                            # ① 종양 찾기 결과 3장 (원본 / 의사 정답 / AI 답)
for k, i in enumerate([3, 120, 240]):
    plt.subplot(1, 3, k+1); draw_one(Xt[i, 0], Yt[i, 0], all_preds[i]); plt.title(f"Dice {all_dice[i]:.0f}%", fontsize=9)
plt.suptitle("Brain MRI tumor segmentation by a small U-Net.  red = doctor's ground truth,  cyan = AI prediction", fontsize=9)
fig.savefig("output/ai_result.png", dpi=130, bbox_inches="tight"); plt.close(fig)

fig = plt.figure(figsize=(12, 2.4))                           # ② 화가 AI가 만든 MRI 6장
for i in range(len(fakes)):
    plt.subplot(1, len(fakes), i+1); plt.imshow(fakes[i, 0], cmap="gray", vmin=-1, vmax=1); plt.axis("off")
plt.suptitle("Synthetic brain MRI slices generated from pure noise by a small diffusion model (128x128)", fontsize=9)
fig.savefig("output/generated_mri.png", dpi=130, bbox_inches="tight"); plt.close(fig)
print("저장 완료:", os.path.abspath("output/ai_result.png"), "/", os.path.abspath("output/generated_mri.png"))

QUESTIONS = [
    "이 그림은 고등학생들과 함께 만든 작은 AI가 공개 뇌 MRI 데이터에서 종양을 찾은 결과야. 빨간 선이 의사가 그린 정답, 하늘색이 AI의 답이야. AI가 얼마나 잘했는지 평가해 주고, 실제 병원에서 이런 AI를 쓸 때 조심해야 할 점을 고등학생 눈높이로 5문장 이내로 설명해 줘.",
    "이 사진들은 AI가 노이즈에서 만들어 낸, 세상에 없는 가짜 뇌 MRI야. 진짜 MRI와 비교해서 어디가 그럴듯하고 어디가 어색한지 짚어 주고, 이런 가짜 의료 영상이 연구에 어떻게 쓰일 수 있는지 3가지만 알려 줘.",
    "의료영상 AI 연구자가 되려면 고등학생 때 무엇을 공부하면 좋을까? 3가지만 짧게 알려 줘.",
]
print("\n복사해서 쓸 질문:\n"); [print(f"Q{k+1}. {q}\n") for k, q in enumerate(QUESTIONS)]

if "google.colab" in sys.modules:                               # Colab이면 그림 파일을 내 컴퓨터로 내려받기
    from google.colab import files
    files.download("output/ai_result.png"); files.download("output/generated_mri.png")

### 🖥️ 브라우저에서 하는 방법 (1분)
1. **챗봇 열기** — [chatgpt.com](https://chatgpt.com) 또는 [gemini.google.com](https://gemini.google.com) (둘 다 무료, 설치 없음)
2. **그림 넣기** — 방법 A: 위에서 저장된 `output/ai_result.png`를 입력창에 **드래그**. 방법 B (더 빠름): `Win + Shift + S`로 화면의 결과 그림을 캡처한 뒤 입력창에 `Ctrl + V`
3. **질문 붙여넣기** — 위 출력의 Q1을 복사해서 붙여 넣고 Enter. 이어서 `generated_mri.png` + Q2, 그리고 학생 질문

> 🙋 **학생들도 함께**: 휴대폰으로 화면의 결과 그림을 **찍어서** ChatGPT·Gemini 앱에 올리고 자기만의 질문을 해 보세요. 답이 서로 다를 수 있습니다. 왜 다를까요?
>
> 💬 LLM은 사진을 "보고" 그럴듯하게 설명하지만, **의사도 아니고 틀릴 수도 있습니다.** AI의 답을 사람이 검토하는 것, 오늘 계속 나온 이야기입니다.

---
## 오늘 배운 것 정리

| | 사람이 하는 일 | AI가 하는 일 |
|---|---|---|
| **1. 사진 = 숫자** | 의사가 3,064장에 종양 테두리를 그린다 (정답 만들기) | 숫자표 16,384개를 읽는다 |
| **2. 규칙 vs 학습** | 규칙 4개를 겹쳐 봐도 평균 점수가 낮다 | — |
| **3. 종양 찾기** | 정답 예시 2,500장을 준다 | 스스로 규칙을 찾아 종양을 찍는다. 오래 배울수록 정확해진다 |
| **4. MRI 그리기** | 사진에 노이즈를 섞는다 | 노이즈를 걷어내는 법을 배워 새 MRI를 만든다 |
| **5. LLM** | 질문하고, 답을 검토한다 | 사진을 보고 말로 설명한다 |

**AI는 규칙을 외우는 게 아니라, 예시에서 규칙을 스스로 찾아냅니다.**
그래서 좋은 **데이터**(정답이 정확한 사진)와, 결과를 **검증하는 사람**(의사·연구자)이 꼭 필요합니다.

> 오늘 쓴 코드는 전부 150줄이 안 됩니다. 데이터도 공개되어 있으니 여러분도 집에서 똑같이 해볼 수 있어요! 🚀

---
<small>데이터 출처: Cheng, J. et al. (2015). *brain tumor dataset*. figshare. https://doi.org/10.6084/m9.figshare.1512427 (CC BY 4.0)</small>